# Tic-Tac-Toe Structural Estimation (Robust)

End-to-end: data generation → MLE (Newton–Armijo) → evaluation → visualization → human-vs-bot.
Only dependency is `numpy`.


In [2]:
import numpy as np
from pathlib import Path
from tictactoe_play_utils import (FEATURE_NAMES, features, logits_for_state,
    play_game, board_to_str, print_trajectory, human_vs_bot,
    X, O)
np.set_printoptions(precision=3, suppress=True)

## Likelihood, Gradient, Hessian

In [3]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    z -= np.max(z)
    ez = np.exp(z); s = ez.sum()
    return ez/s

def loglik(data, theta, lam=0.0):
    ll = 0.0
    for board, player, action in data:
        z, moves, Xmat = logits_for_state(board, player, theta)
        if not moves: continue
        p = softmax(z)
        j = moves.index(action)
        ll += np.log(max(p[j], 1e-12))
    return ll - 0.5*lam*np.dot(theta,theta)

def grad_hess(data, theta, lam=0.0):
    K = len(theta)
    g = np.zeros(K)
    H = np.zeros((K,K))
    for board, player, action in data:
        z, moves, Xmat = logits_for_state(board, player, theta)
        if not moves: continue
        p = softmax(z)
        j = moves.index(action)
        xj = Xmat[j]
        Ex = p @ Xmat
        g += xj - Ex
        Exx = (Xmat.T * p) @ Xmat
        H -= Exx - np.outer(Ex, Ex)
    g -= lam*theta
    H -= lam*np.eye(K)
    return g, H

def newton_maximize(data, theta0, lam=1e-6, max_iter=50, tol=1e-6, armijo=1e-4, backtrack=0.5):
    th = theta0.astype(float).copy()
    hist = []
    for it in range(1, max_iter+1):
        g, H = grad_hess(data, th, lam)
        gn = float(np.linalg.norm(g, 2))
        ll = loglik(data, th, lam)
        hist.append((it, ll, gn, th.copy()))
        print(f"iter {it:02d}: ll={ll:.2f} ||g||={gn:.3e} theta={th}")
        if gn < tol: break
        try:
            step = np.linalg.solve(-H, g)
        except np.linalg.LinAlgError:
            step = np.linalg.solve(-(H - 1e-6*np.eye(len(th))), g)
        t, base = 1.0, ll
        while True:
            th_new = th + t*step
            ll_new = loglik(data, th_new, lam)
            if ll_new >= base + armijo*t*float(g @ step):
                th = th_new; break
            t *= backtrack
            if t < 1e-8:
                th = th_new; break
    return th, loglik(data, th, lam), hist

## Generate dataset

In [4]:
from tictactoe_play_utils import sample_policy_move, other, check_winner, apply_move

def generate_dataset(n_games=1500, theta_star=None, seed=42):
    if theta_star is None:
        theta_star = np.array([3.0, 2.0, 0.5, 0.3, 0.0, 0.6])
    rng = np.random.default_rng(seed)
    data, wins = [], {X:0, O:0, 0:0}
    for _ in range(n_games):
        first = X if rng.random()<0.5 else O
        board = tuple([0]*9); player = first
        while True:
            w, draw = check_winner(board)
            if w!=0 or draw:
                wins[w]+=1; break
            mv = sample_policy_move(board, player, theta_star, rng)
            if mv is None: wins[0]+=1; break
            data.append((board, player, mv))
            board = apply_move(board, mv, player)
            player = other(player)
    return data, wins, theta_star

In [5]:
dataset, wins_gt, theta_star = generate_dataset(n_games=1500, theta_star=np.array([1.0,5.0,1.5,0.3,0.0,0.6]))
print('Generated', len(dataset), 'state-action pairs.')
print('Wins (X,O,draw)=', wins_gt.get(X,0), wins_gt.get(O,0), wins_gt.get(0,0))

Generated 13064 state-action pairs.
Wins (X,O,draw)= 140 146 1214


## Estimate θ

In [6]:
theta0 = np.zeros(len(FEATURE_NAMES))
theta_hat, ll, hist = newton_maximize(dataset, theta0, lam=1e-6)
print('\nEstimated θ:', theta_hat)
print('True θ*    :', theta_star)
print('Final ll   :', ll)

iter 01: ll=-19046.99 ||g||=4.166e+03 theta=[0. 0. 0. 0. 0. 0.]
iter 02: ll=-11895.04 ||g||=6.005e+02 theta=[ 0.587  3.622  1.178 -0.5   -0.678  0.339]
iter 03: ll=-11644.19 ||g||=9.563e+01 theta=[ 0.949  4.435  0.911 -0.328 -0.583  0.578]
iter 04: ll=-11619.17 ||g||=1.684e+01 theta=[ 1.027  4.916  0.897 -0.316 -0.581  0.64 ]
iter 05: ll=-11617.98 ||g||=1.035e+00 theta=[ 1.033  5.052  0.897 -0.316 -0.581  0.646]
iter 06: ll=-11617.97 ||g||=4.530e-03 theta=[ 1.033  5.061  0.897 -0.316 -0.581  0.646]
iter 07: ll=-11617.97 ||g||=8.652e-08 theta=[ 1.033  5.062  0.897 -0.316 -0.581  0.646]

Estimated θ: [ 1.033  5.062  0.897 -0.316 -0.581  0.646]
True θ*    : [1.  5.  1.5 0.3 0.  0.6]
Final ll   : -11617.974138619204


## Visualize a sample trajectory (3×3 boards)

In [10]:
traj, w = play_game(theta_star, first_player=X, seed=0, stochastic=False)
print_trajectory(traj)
print('Winner:', {0:'draw',1:'X',2:'O'}[w])

# for i in range(0,10):
#     traj, w = play_game(theta_star, first_player=X, seed=0, stochastic=True)
#     print(f'Game {i} --- Winner:', {0:'draw',1:'X',2:'O'}[w])

Move 1: Player X -> 4
. . .
. . .
. . .

Move 2: Player O -> 0
. . .
. X .
. . .

Move 3: Player X -> 2
O . .
. X .
. . .

Move 4: Player O -> 6
O . X
. X .
. . .

Move 5: Player X -> 3
O . X
. X .
O . .

Move 6: Player O -> 5
O . X
X X .
O . .

Move 7: Player X -> 1
O . X
X X O
O . .

Move 8: Player O -> 7
O X X
X X O
O . .

Move 9: Player X -> 8
O X X
X X O
O O .

Winner: draw


## Play against the bot

In [ ]:
## Uncomment to play in the notebook console
# human_vs_bot(theta_hat, bot_player='O', stochastic=False)

## Export CSV

In [ ]:
import csv
def board_to_string(board): return ''.join('.XO'[v] for v in board)
csv_path = Path('tictactoe_dataset.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['board','player','action'])
    for b,p,a in dataset: w.writerow([board_to_string(b), p, a])
print('Wrote CSV to', csv_path.resolve())